# Graph Multiset Transformer (GMT) Pooling on PROTEINS

Graph Classification on PROTEINS (TUDataset): Multi-head attention pooling capturing inter-node interactions across whole graphs. This notebook implements the approach with `GraphMultisetTransformer` inside a `K3GMTNet` model, evaluating the result on held-out data. The single code cell below installs **K3-Node**, loads the dataset, defines the model using K3-Node's `GraphMultisetTransformer` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install -q torch_geometric
!pip install git+http://github.com/anas-rz/k3-node/@examples-check

# ==============================================================================
# Part 2: K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, ops

import k3_node
from k3_node import layers as k3_layers
from k3_node.datasets import TUDataset

title = "Graph Multiset Transformer (GMT) Pooling on PROTEINS"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. Dataset
dataset = TUDataset(root="./data/PROTEINS", name="PROTEINS")

# 2. GMT Model Definition
class K3GMTNet(keras.Model):
    def __init__(self, in_channels, hidden_channels, out_channels=2):
        super().__init__()
        self.conv1 = k3_layers.GCNConv(in_channels, hidden_channels)
        self.conv2 = k3_layers.GCNConv(hidden_channels, hidden_channels)
        self.pool = k3_layers.GraphMultisetTransformer(hidden_channels, 4, hidden_channels)
        self.lin = layers.Dense(out_channels)

    def call(self, x, edge_index, batch=None):
        x = ops.relu(self.conv1(x, edge_index))
        x = ops.relu(self.conv2(x, edge_index))
        out = self.pool(x, batch=batch)
        return self.lin(out)

k3_model = K3GMTNet(dataset.num_features, 32, dataset.num_classes)

# 3. Forward Pass Test
num_nodes = 40
dummy_x = keras.random.normal((num_nodes, dataset.num_features))
dummy_edges = ops.convert_to_tensor([[0, 1], [1, 0]], dtype="int64")
dummy_batch = ops.zeros((num_nodes,), dtype="int64")

out = k3_model(dummy_x, dummy_edges, dummy_batch)
print(f"GMT forward pass output shape: {out.shape}")

print("\n✓ K3-Node GMT execution completed successfully!")